# 05. 벡터 스토어 (Vector Store)

임베딩 벡터를 저장하고, 쿼리와 **가장 유사한 문서를 빠르게 검색**합니다.

## FAISS란?
Facebook AI Research에서 만든 벡터 유사도 검색 라이브러리입니다.  
로컬에서 빠르게 동작하며 실습에 적합합니다.

In [8]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 준비
loader = TextLoader("data/ai_basic.txt", encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"청크 수: {len(chunks)}")

청크 수: 10


## 1. FAISS 벡터 스토어 생성

`from_documents()`는 청크를 임베딩하고 벡터 스토어에 저장합니다.

## 방법 1. 거리 기반 유사도 계산방식
### 1. 벡터 스토어 생성

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

# 거리 기반 매트릭 설정
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
    # distance_strategy=DistanceStrategy.EUCLIDEAN # <-- 유클리드 거리 기준 (기본값)
    distance_strategy=DistanceStrategy.COSINE,
)

### 2. 유사도 검색 - similarity_search()

In [24]:
query = "RAG란 무엇인가요?"

results = vector_store.similarity_search(query, k=3)

print(f"검색 결과 {len(results)}개\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content[:150]}")
    print(f"     출처: {doc.metadata}")
    print()

검색 결과 3개

[1] RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시
     출처: {'source': 'data/ai_basic.txt'}

[2] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다
     출처: {'source': 'data/ai_basic.txt'}

[3] 트랜스포머(Transformer)란?
트랜스포머는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져왔습니다. 어텐션 메커니즘(Attention Mechanism)을 핵심으로 사용하며, 병렬 처리가 가능해 학습 속도가 빠릅니다. BERT, GPT 
     출처: {'source': 'data/ai_basic.txt'}



### 3. 유사도 점수 포함 검색 - similarity_search_with_score()

In [25]:
results_with_score = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results_with_score:
    print(f"거리 점수: {score:.4f}")
    print(f"내용: {doc.page_content[:100]}")
    print()

거리 점수: 0.8905
내용: RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 

거리 점수: 1.4583
내용: LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공

거리 점수: 1.5483
내용: 트랜스포머(Transformer)란?
트랜스포머는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져왔습니다. 어텐션 메커니즘(Attention Mechani



## 방법 2. FAISS Native 방식
| 목적       | FAISS 인덱스                | 정규화                 | score 해석 |
| -------- | ------------------------ | ------------------- | -------- |
| 유클리디언 거리 | `faiss.IndexFlatL2(dim)` | 필요 없음               | 낮을수록 유사  |
| 코사인 유사도  | `faiss.IndexFlatIP(dim)` | `normalize_L2=True` | 높을수록 유사  |


### 1. IndexFlatIP(코사인 유사도) 기반 검색 방식

#### 1) 벡터 스토어 생성

In [27]:
# FAISS 코사인유사도 인덱싱 방식
# 
import faiss
from langchain_community.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# 1. 임베딩 차원 확인
dim = len(embeddings.embed_query("test"))

# 2. FAISS Inner Product 인덱스 생성
# 정규화된 벡터에서는 Inner Product가 Cosine Similarity처럼 동작한다.
index = faiss.IndexFlatIP(dim)

# 3. LangChain FAISS VectorStore 생성
vector_store_ip = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
    normalize_L2=True,
)

# 4. chunks 문서를 vector_store에 추가 
vector_store_ip.add_documents(chunks)

print("벡터 스토어 생성 완료")
print(f"저장된 벡터 수: {vector_store_ip.index.ntotal}")

벡터 스토어 생성 완료
저장된 벡터 수: 10


#### 2) 유사도 검색

In [28]:
query = "RAG란 무엇인가요?"

results = vector_store_ip.similarity_search(query, k=3)

print(f"검색 결과 {len(results)}개\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content[:150]}")
    print(f"     출처: {doc.metadata}")
    print()

검색 결과 3개

[1] RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시
     출처: {'source': 'data/ai_basic.txt'}

[2] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다
     출처: {'source': 'data/ai_basic.txt'}

[3] 트랜스포머(Transformer)란?
트랜스포머는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져왔습니다. 어텐션 메커니즘(Attention Mechanism)을 핵심으로 사용하며, 병렬 처리가 가능해 학습 속도가 빠릅니다. BERT, GPT 
     출처: {'source': 'data/ai_basic.txt'}



#### 3) 점수 포함 검색 - FAISS Native 코사인 유사도 

In [33]:
results_with_score = vector_store_ip.similarity_search_with_score(query, k=3)

for doc, score in results_with_score:
    print(f"거리 점수: {score:.4f}")
    print(f"내용: {doc.page_content[:100]}")
    print()

거리 점수: 0.5550
내용: RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 

거리 점수: 0.2706
내용: LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공

거리 점수: 0.2262
내용: 트랜스포머(Transformer)란?
트랜스포머는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져왔습니다. 어텐션 메커니즘(Attention Mechani



> [해석] FAISS Native 코사인 유사도 기반 검색은 값이 높을 수록 유사도가 높음

### 2. L2(유클리디언) 거리 기반 검색 방식

#### 1) 벡터 스토어 생성

In [30]:
# 유클리디언 거리 방식
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

# 1. 임베딩 차원 확인
dim = len(embeddings.embed_query("test"))

# 2. FAISS L2 인덱스 생성
#    L2 = Euclidean distance
index = faiss.IndexFlatL2(dim)

# 3. LangChain FAISS VectorStore 생성
vector_store_l2 = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

# 4. 문서 추가
vector_store_l2.add_documents(chunks)
print("벡터 스토어 생성 완료")
print(f"저장된 벡터 수: {vector_store_l2.index.ntotal}")

벡터 스토어 생성 완료
저장된 벡터 수: 10


#### 2) 유사도 검색 - similarity_search()

In [31]:
query = "RAG란 무엇인가요?"

results = vector_store.similarity_search(query, k=3)

print(f"검색 결과 {len(results)}개\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content[:150]}")
    print(f"     출처: {doc.metadata}")
    print()

검색 결과 3개

[1] RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시
     출처: {'source': 'data/ai_basic.txt'}

[2] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다
     출처: {'source': 'data/ai_basic.txt'}

[3] 트랜스포머(Transformer)란?
트랜스포머는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져왔습니다. 어텐션 메커니즘(Attention Mechanism)을 핵심으로 사용하며, 병렬 처리가 가능해 학습 속도가 빠릅니다. BERT, GPT 
     출처: {'source': 'data/ai_basic.txt'}



#### 3) 유사도 점수 포함 검색 - similarity_search_with_score()

In [32]:
results_with_score = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results_with_score:
    print(f"거리 점수: {score:.4f}")
    print(f"내용: {doc.page_content[:100]}")
    print()

거리 점수: 0.8905
내용: RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 

거리 점수: 1.4583
내용: LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공

거리 점수: 1.5483
내용: 트랜스포머(Transformer)란?
트랜스포머는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져왔습니다. 어텐션 메커니즘(Attention Mechani



> [해석] FAISS native로 거리기반 유사도는 score가 작을 수록 유사함

## 4. 벡터 스토어 저장 및 불러오기

In [5]:
# 로컬에 저장
vector_store.save_local("data/faiss_index")
print("저장 완료")

저장 완료


In [6]:
# 저장된 인덱스 불러오기
loaded_store = FAISS.load_local(
    "data/faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

# 검색 테스트
results = loaded_store.similarity_search("임베딩이 뭔가요?", k=2)
for doc in results:
    print(doc.page_content[:150])
    print()

임베딩(Embedding)이란?
임베딩은 텍스트를 숫자 벡터로 변환하는 기술입니다. 의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다. OpenAI의 text-embedding-ada-002, text-embedding-3-small 등이 널리 사용됩니다.

벡터 스토어(Vector Store)란?
벡터 스토어는 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma, Pinecone, Weaviate 등이 대표적입니다. 코사인 유사도나 내적을 사용해 쿼리와 가장 유사한 문서를 빠르게 찾



## 5. 문서 추가

In [7]:
from langchain_core.documents import Document

new_docs = [
    Document(
        page_content="LangSmith는 LLM 애플리케이션의 디버깅과 모니터링 도구입니다.",
        metadata={"source": "manual"}
    )
]

vector_store.add_documents(new_docs)
print(f"추가 후 벡터 수: {vector_store.index.ntotal}")

추가 후 벡터 수: 11


## 정리

| 메서드 | 설명 |
|---|---|
| `from_documents()` | 문서 임베딩 후 저장 |
| `similarity_search()` | 유사 문서 검색 |
| `similarity_search_with_score()` | 유사도 점수 포함 검색 |
| `save_local()` / `load_local()` | 저장/불러오기 |
| `add_documents()` | 문서 추가 |

다음 단계: **Retriever** →